
# STEP 34 — Table 1, Table 2, and Table 3 Auto Generator  
## BrainFMOps-Analyze Publication Edition

Notebook นี้สร้างตารางสำหรับบทความโดยอัตโนมัติจากไฟล์ 2 ชุดในโฟลเดอร์:

```text
<repository-root>\34_Table_Generator_Input
```

ไฟล์ที่ต้องมี:

```text
evaluation_summary_with_labels.csv
oasis_cross-sectional*.xlsx
```

ผลลัพธ์จะถูกสร้างใน:

```text
<repository-root>\34_Table_Generator_Output
```

Notebook ทำงานดังนี้:

1. ตรวจและล็อก evaluation cohort
2. Merge subject-level predictions กับ OASIS clinical spreadsheet
3. สร้าง **Table 1** พร้อม p-values
4. สร้าง **Table 2** จาก model configuration และ environment
5. สร้าง **Table 3** พร้อม bootstrap 95% CI
6. สร้าง merge audit, missing-data report, CSV, Excel และ JSON manifest

> เปิด Notebook แล้วเลือก **Kernel → Restart & Run All**


In [ ]:

from pathlib import Path
import json
import math
import platform
import sys
import warnings

import numpy as np
import pandas as pd

from scipy import stats
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# 1) Project paths
# ------------------------------------------------------------------
ROOT_CANDIDATES = [
    Path.cwd().resolve(),
    Path.cwd().resolve(),
    Path.cwd(),
]

ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), Path.cwd())
INPUT_DIR = ROOT / "34_Table_Generator_Input"
OUTPUT_DIR = ROOT / "34_Table_Generator_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_FILE = INPUT_DIR / "evaluation_summary_with_labels.csv"
CLINICAL_FILES = sorted(
    list(INPUT_DIR.glob("oasis_cross-sectional*.xlsx"))
    + list(INPUT_DIR.glob("oasis_cross-sectional*.xls"))
    + list(INPUT_DIR.glob("oasis_cross-sectional*.csv"))
)

if not PREDICTION_FILE.exists():
    raise FileNotFoundError(f"ไม่พบไฟล์: {PREDICTION_FILE}")

if not CLINICAL_FILES:
    raise FileNotFoundError(
        "ไม่พบไฟล์ OASIS clinical spreadsheet ที่ขึ้นต้นด้วย "
        "'oasis_cross-sectional' ในโฟลเดอร์ input"
    )

CLINICAL_FILE = CLINICAL_FILES[0]

print("ROOT            :", ROOT)
print("PREDICTION FILE :", PREDICTION_FILE)
print("CLINICAL FILE   :", CLINICAL_FILE)
print("OUTPUT DIR      :", OUTPUT_DIR)


In [ ]:

# ------------------------------------------------------------------
# 2) Publication configuration
#    แก้เฉพาะค่าที่เป็น UNKNOWN ก่อนส่งบทความ
# ------------------------------------------------------------------
CONFIG = {
    "operating_threshold": 0.32,
    "bootstrap_iterations": 5000,
    "bootstrap_seed": 42,

    # Model configuration for Table 2
    "model_name_architecture": "UNKNOWN — fill from inference code",
    "pretraining_corpus": "UNKNOWN — fill from model documentation",
    "checkpoint_source_version": "UNKNOWN — fill checkpoint path/version",
    "number_of_parameters": "UNKNOWN",
    "rationale_for_model_selection": "Frozen pretrained baseline for reproducible pipeline evaluation",
    "task_specific_fine_tuning": "None; backbone weights frozen",
    "classification_head": "UNKNOWN — specify linear head / zero-shot / other",
    "input_resolution_channels": "UNKNOWN — specify resolution and channel handling",
    "intensity_normalization": "UNKNOWN — specify scaling and mean/SD",
    "slice_selection": "UNKNOWN — specify all axial slices or selected slices",
    "aggregation_over_slices": "Arithmetic mean of slice-level probabilities",
    "software_framework": "Auto-detected below",
    "hardware": "Auto-detected below",
    "random_seed": 42,

    # Label definition
    "cn_definition": "Ground-truth label CN in evaluation file",
    "ad_definition": "Ground-truth label AD in evaluation file",
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


In [ ]:

# ------------------------------------------------------------------
# 3) Helper functions
# ------------------------------------------------------------------
def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path}")

def first_existing(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None

def best_existing_column(df, candidates, normalizer=None, numeric=False):
    """Select the candidate column with the largest number of usable values."""
    lookup = {str(c).strip().lower(): c for c in df.columns}
    scored = []
    for candidate in candidates:
        actual = lookup.get(candidate.lower())
        if actual is None:
            continue
        series = df[actual]
        if numeric:
            usable = pd.to_numeric(series, errors="coerce").notna().sum()
        elif normalizer is not None:
            usable = series.map(normalizer).notna().sum()
        else:
            usable = series.notna().sum()
        scored.append((int(usable), actual))
    if not scored:
        return None
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[0][1]

def normalize_subject_id(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().upper()
    text = text.replace("-", "_").replace(" ", "_")
    while "__" in text:
        text = text.replace("__", "_")
    return text

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().upper()

    positive = {
        "1", "AD", "ALZHEIMER", "ALZHEIMER'S DISEASE",
        "DEMENTED", "POSITIVE", "TRUE"
    }
    negative = {
        "0", "CN", "CONTROL", "COGNITIVELY NORMAL",
        "NONDEMENTED", "NEGATIVE", "FALSE", "NORMAL"
    }

    if text in positive:
        return 1
    if text in negative:
        return 0

    try:
        number = float(text)
        if number == 1:
            return 1
        if number == 0:
            return 0
    except Exception:
        pass

    return np.nan

def format_pvalue(p):
    if p is None or pd.isna(p):
        return "—"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

def mean_sd(series):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    return f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

def median_iqr(series):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    q1, med, q3 = np.percentile(x, [25, 50, 75])
    return f"{med:.1f} ({q1:.1f}–{q3:.1f})"

def n_percent(n, total):
    if total == 0:
        return "0 (0.0%)"
    return f"{int(n)} ({100*n/total:.1f}%)"

def welch_p(cn, ad):
    cn = pd.to_numeric(cn, errors="coerce").dropna()
    ad = pd.to_numeric(ad, errors="coerce").dropna()
    if len(cn) < 2 or len(ad) < 2:
        return np.nan
    return stats.ttest_ind(cn, ad, equal_var=False, nan_policy="omit").pvalue

def mann_whitney_p(cn, ad):
    cn = pd.to_numeric(cn, errors="coerce").dropna()
    ad = pd.to_numeric(ad, errors="coerce").dropna()
    if len(cn) == 0 or len(ad) == 0:
        return np.nan
    return stats.mannwhitneyu(cn, ad, alternative="two-sided").pvalue

def categorical_p(table):
    table = np.asarray(table, dtype=float)
    if table.size == 0 or table.sum() == 0:
        return np.nan, "NA"
    chi2, p, dof, expected = stats.chi2_contingency(table)
    if table.shape == (2, 2) and (expected < 5).any():
        _, p = stats.fisher_exact(table)
        return p, "Fisher exact"
    return p, "Chi-square"

def percentile_ci(values, alpha=0.05):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan
    return tuple(np.percentile(arr, [100*alpha/2, 100*(1-alpha/2)]))


In [ ]:

# ------------------------------------------------------------------
# 4) Load prediction and clinical data
# ------------------------------------------------------------------
pred_raw = load_table(PREDICTION_FILE)
clinical_raw = load_table(CLINICAL_FILE)

print("Prediction shape:", pred_raw.shape)
print("Clinical shape  :", clinical_raw.shape)

print("\nPrediction columns:")
print(list(pred_raw.columns))

print("\nClinical columns:")
print(list(clinical_raw.columns))


In [ ]:

# ------------------------------------------------------------------
# 5) Detect prediction columns and lock labelled cohort
# ------------------------------------------------------------------
PRED_ID_CANDIDATES = [
    "subject_key", "subject_id", "case_id", "id", "subject"
]
TRUE_CANDIDATES = [
    "ground_truth_original", "ground_truth_derived", "ground_truth",
    "true_label", "label", "diagnosis", "class", "target", "y_true"
]
PROB_CANDIDATES = [
    "probability_positive", "subject_probability",
    "subject_level_probability", "ad_probability",
    "predicted_ad_probability", "mean_probability",
    "probability", "prediction_probability", "y_prob", "score"
]
PRED_LABEL_CANDIDATES = [
    "prediction", "predicted_label", "subject_prediction", "y_pred"
]
SLICE_COUNT_CANDIDATES = [
    "n_slices", "slice_count", "slices_analysed", "slices_analyzed",
    "selected_slice_count", "num_slices", "number_of_slices"
]

pred_id_col = best_existing_column(pred_raw, PRED_ID_CANDIDATES)
true_col = best_existing_column(
    pred_raw, TRUE_CANDIDATES, normalizer=normalize_binary_label
)
prob_col = best_existing_column(
    pred_raw, PROB_CANDIDATES, numeric=True
)
pred_label_col = best_existing_column(pred_raw, PRED_LABEL_CANDIDATES)
slice_count_col = best_existing_column(
    pred_raw, SLICE_COUNT_CANDIDATES, numeric=True
)

if pred_id_col is None:
    raise KeyError("ไม่พบ subject ID column ใน prediction file")
if true_col is None:
    raise KeyError("ไม่พบ ground-truth column ใน prediction file")
if prob_col is None:
    raise KeyError("ไม่พบ probability column ใน prediction file")

pred = pred_raw.copy()
pred["_subject_id"] = pred[pred_id_col].map(normalize_subject_id)
pred["_y_true"] = pred[true_col].map(normalize_binary_label)
pred["_y_prob"] = pd.to_numeric(pred[prob_col], errors="coerce")

print("Usable values before filtering:")
print(" Subject IDs :", int(pred["_subject_id"].notna().sum()))
print(" Valid labels:", int(pred["_y_true"].notna().sum()))
print(" Probabilities:", int(pred["_y_prob"].notna().sum()))

pred = pred[
    pred["_subject_id"].notna()
    & pred["_y_true"].notna()
    & pred["_y_prob"].notna()
    & pred["_y_prob"].between(0, 1)
].copy()

if len(pred) == 0:
    raise ValueError(
        "ไม่พบ evaluation cohort หลังการกรองข้อมูล\n"
        f"Selected ID column: {pred_id_col}\n"
        f"Selected label column: {true_col}\n"
        f"Selected probability column: {prob_col}\n"
        "กรุณาตรวจ output ด้านบนว่าคอลัมน์ใดมี usable values เป็นศูนย์"
    )

pred["_y_true"] = pred["_y_true"].astype(int)
pred["_group"] = pred["_y_true"].map({0: "CN", 1: "AD"})
pred["_y_pred"] = (
    pred["_y_prob"] >= CONFIG["operating_threshold"]
).astype(int)

if slice_count_col is not None:
    pred["_slice_count"] = pd.to_numeric(
        pred[slice_count_col], errors="coerce"
    )
else:
    pred["_slice_count"] = np.nan

duplicate_pred = pred[pred["_subject_id"].duplicated(keep=False)].copy()

if not duplicate_pred.empty:
    print("WARNING: duplicate subject IDs found in prediction file")
    display(duplicate_pred[["_subject_id", "_group", "_y_prob"]].head(20))

pred = pred.drop_duplicates("_subject_id", keep="first").copy()

print("Detected columns:")
print(" Subject ID  :", pred_id_col)
print(" Ground truth:", true_col)
print(" Probability :", prob_col)
print(" Prediction  :", pred_label_col)
print(" Slice count :", slice_count_col)

print("\nLocked evaluation cohort:")
print(pred["_group"].value_counts())
print("Total:", len(pred))


In [ ]:

# ------------------------------------------------------------------
# 6) Detect clinical columns and merge
# ------------------------------------------------------------------
CLINICAL_CANDIDATES = {
    "clinical_id": ["ID", "subject_id", "subject", "subject_key"],
    "sex": ["M/F", "sex", "gender"],
    "age": ["Age", "age_years"],
    "education": ["Educ", "education", "education_years"],
    "mmse": ["MMSE", "mmse_score"],
    "cdr": ["CDR", "cdr_score"],
    "etiv": ["eTIV", "etiv", "estimated_total_intracranial_volume"],
    "nwbv": ["nWBV", "nwbv", "normalized_whole_brain_volume"],
}

clinical_cols = {
    key: first_existing(clinical_raw.columns, candidates)
    for key, candidates in CLINICAL_CANDIDATES.items()
}

if clinical_cols["clinical_id"] is None:
    raise KeyError("ไม่พบ ID column ใน OASIS clinical spreadsheet")

clinical = clinical_raw.copy()
clinical["_subject_id"] = clinical[
    clinical_cols["clinical_id"]
].map(normalize_subject_id)

duplicate_clinical = clinical[
    clinical["_subject_id"].duplicated(keep=False)
].copy()

clinical_unique = clinical.drop_duplicates(
    "_subject_id", keep="first"
).copy()

merged = pred.merge(
    clinical_unique,
    on="_subject_id",
    how="left",
    suffixes=("_pred", "_clinical"),
    indicator=True,
)

print("Clinical column mapping:")
for key, value in clinical_cols.items():
    print(f" {key:12s}: {value}")

print("\nMerge audit:")
print(merged["_merge"].value_counts())


In [ ]:

# ------------------------------------------------------------------
# 7) Save merge audit and missing-data reports
# ------------------------------------------------------------------
if len(merged) == 0:
    raise ValueError(
        "Merged dataframe มี 0 แถว เพราะ evaluation cohort ว่าง\n"
        "ย้อนตรวจ Cell 5: Valid labels และ Probabilities ต้องมากกว่า 0"
    )
merge_audit = pd.DataFrame({
    "Item": [
        "Prediction subjects after label filtering",
        "Matched with clinical spreadsheet",
        "Unmatched prediction subjects",
        "Duplicate prediction IDs before deduplication",
        "Duplicate clinical IDs before deduplication",
        "CN subjects",
        "AD subjects",
    ],
    "Value": [
        len(pred),
        int((merged["_merge"] == "both").sum()),
        int((merged["_merge"] != "both").sum()),
        int(duplicate_pred["_subject_id"].nunique()),
        int(duplicate_clinical["_subject_id"].nunique()),
        int((pred["_group"] == "CN").sum()),
        int((pred["_group"] == "AD").sum()),
    ]
})

unmatched = merged.loc[
    merged["_merge"] != "both",
    ["_subject_id", "_group", "_y_prob", "_merge"]
].copy()

missing_rows = []
for variable, column in clinical_cols.items():
    if variable == "clinical_id":
        continue
    if column is None:
        missing_rows.append({
            "Variable": variable,
            "Detected column": "NOT FOUND",
            "Missing n": len(merged),
            "Missing %": 100.0,
        })
    else:
        missing_n = int(merged[column].isna().sum())
        missing_rows.append({
            "Variable": variable,
            "Detected column": column,
            "Missing n": missing_n,
            "Missing %": round(100 * missing_n / len(merged), 2) if len(merged) else np.nan,
        })

missing_report = pd.DataFrame(missing_rows)

display(merge_audit)
display(missing_report)


In [ ]:

# ------------------------------------------------------------------
# 8) Build Table 1: Demographic and clinical characteristics
# ------------------------------------------------------------------
analysis = merged[merged["_merge"] == "both"].copy()
cn = analysis[analysis["_group"] == "CN"].copy()
ad = analysis[analysis["_group"] == "AD"].copy()

def get_series(df, key):
    col = clinical_cols.get(key)
    if col is None:
        return pd.Series(dtype=float)
    return df[col]

table1_rows = []

# Age
if clinical_cols["age"] is not None:
    p = welch_p(get_series(cn, "age"), get_series(ad, "age"))
    table1_rows.append([
        "Age (years), mean ± SD",
        mean_sd(get_series(cn, "age")),
        mean_sd(get_series(ad, "age")),
        mean_sd(get_series(analysis, "age")),
        format_pvalue(p),
        "Welch t-test",
    ])

# Sex
if clinical_cols["sex"] is not None:
    sex_col = clinical_cols["sex"]
    sex_norm = analysis[sex_col].astype(str).str.strip().str.upper()
    analysis["_sex_norm"] = sex_norm.replace({
        "FEMALE": "F", "MALE": "M", "WOMAN": "F", "MAN": "M"
    })
    cn = analysis[analysis["_group"] == "CN"].copy()
    ad = analysis[analysis["_group"] == "AD"].copy()

    cn_f = int((cn["_sex_norm"] == "F").sum())
    cn_m = int((cn["_sex_norm"] == "M").sum())
    ad_f = int((ad["_sex_norm"] == "F").sum())
    ad_m = int((ad["_sex_norm"] == "M").sum())

    p, test_name = categorical_p([[cn_f, cn_m], [ad_f, ad_m]])

    table1_rows.append([
        "Sex, female / male, n (%)",
        f"{n_percent(cn_f, len(cn))} / {n_percent(cn_m, len(cn))}",
        f"{n_percent(ad_f, len(ad))} / {n_percent(ad_m, len(ad))}",
        f"{n_percent(cn_f + ad_f, len(analysis))} / "
        f"{n_percent(cn_m + ad_m, len(analysis))}",
        format_pvalue(p),
        test_name,
    ])

# Continuous variables
for key, label in [
    ("education", "Education (years), mean ± SD"),
    ("mmse", "MMSE, mean ± SD"),
    ("etiv", "eTIV, mean ± SD"),
    ("nwbv", "nWBV, mean ± SD"),
]:
    if clinical_cols[key] is not None:
        p = welch_p(get_series(cn, key), get_series(ad, key))
        table1_rows.append([
            label,
            mean_sd(get_series(cn, key)),
            mean_sd(get_series(ad, key)),
            mean_sd(get_series(analysis, key)),
            format_pvalue(p),
            "Welch t-test",
        ])

# CDR distribution
if clinical_cols["cdr"] is not None:
    cdr_col = clinical_cols["cdr"]
    cdr_numeric = pd.to_numeric(analysis[cdr_col], errors="coerce")

    categories = [
        ("0", lambda x: x == 0),
        ("0.5", lambda x: x == 0.5),
        ("1", lambda x: x == 1),
        ("≥ 2", lambda x: x >= 2),
    ]

    for idx, (name, selector) in enumerate(categories):
        cn_values = pd.to_numeric(cn[cdr_col], errors="coerce")
        ad_values = pd.to_numeric(ad[cdr_col], errors="coerce")
        all_values = pd.to_numeric(analysis[cdr_col], errors="coerce")

        cn_n = int(selector(cn_values).sum())
        ad_n = int(selector(ad_values).sum())
        all_n = int(selector(all_values).sum())

        table1_rows.append([
            f"CDR {name}, n (%)",
            n_percent(cn_n, len(cn)),
            n_percent(ad_n, len(ad)),
            n_percent(all_n, len(analysis)),
            "—",
            "Descriptive only; CDR contributes to label definition",
        ])

# Slices analysed per subject
if analysis["_slice_count"].notna().any():
    p = mann_whitney_p(cn["_slice_count"], ad["_slice_count"])
    table1_rows.append([
        "MRI slices analysed per subject, median (IQR)",
        median_iqr(cn["_slice_count"]),
        median_iqr(ad["_slice_count"]),
        median_iqr(analysis["_slice_count"]),
        format_pvalue(p),
        "Mann–Whitney U",
    ])
else:
    table1_rows.append([
        "MRI slices analysed per subject, median (IQR)",
        "NA", "NA", "NA", "—",
        "Slice-count column was not found in prediction file",
    ])

table1 = pd.DataFrame(
    table1_rows,
    columns=[
        "Characteristic",
        f"CN (n = {len(cn)})",
        f"AD (n = {len(ad)})",
        f"All (n = {len(analysis)})",
        "p-value",
        "Statistical test",
    ],
)

display(table1)


In [ ]:

# ------------------------------------------------------------------
# 9) Auto-detect environment and build Table 2
# ------------------------------------------------------------------
environment = {
    "Python": sys.version.split()[0],
    "Operating system": platform.platform(),
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
}

try:
    import sklearn
    environment["scikit-learn"] = sklearn.__version__
except Exception:
    environment["scikit-learn"] = "Not available"

try:
    import scipy
    environment["SciPy"] = scipy.__version__
except Exception:
    environment["SciPy"] = "Not available"

try:
    import torch
    environment["PyTorch"] = torch.__version__
    environment["CUDA"] = str(torch.version.cuda)
    environment["GPU"] = (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    )
except Exception:
    environment["PyTorch"] = "Not available"
    environment["CUDA"] = "Not available"
    environment["GPU"] = "Not available"

CONFIG["software_framework"] = "; ".join(
    f"{k} {v}" for k, v in environment.items()
    if k not in {"GPU", "Operating system"}
)
CONFIG["hardware"] = environment.get("GPU", "Unknown")

table2 = pd.DataFrame([
    ["Model name and architecture", CONFIG["model_name_architecture"]],
    ["Pretraining corpus", CONFIG["pretraining_corpus"]],
    ["Checkpoint source and version", CONFIG["checkpoint_source_version"]],
    ["Number of parameters", CONFIG["number_of_parameters"]],
    ["Rationale for model selection", CONFIG["rationale_for_model_selection"]],
    ["Task-specific fine-tuning", CONFIG["task_specific_fine_tuning"]],
    ["Classification head", CONFIG["classification_head"]],
    ["Input resolution and channels", CONFIG["input_resolution_channels"]],
    ["Intensity normalization", CONFIG["intensity_normalization"]],
    ["Slice selection per examination", CONFIG["slice_selection"]],
    ["Aggregation over slices", CONFIG["aggregation_over_slices"]],
    ["Decision threshold", f'T = {CONFIG["operating_threshold"]:.2f}'],
    ["Software framework and version", CONFIG["software_framework"]],
    ["Hardware", CONFIG["hardware"]],
    ["Random seed", CONFIG["random_seed"]],
], columns=["Item", "Setting"])

display(table2)


In [ ]:

# ------------------------------------------------------------------
# 10) Primary metrics for Table 3
# ------------------------------------------------------------------
y_true = pred["_y_true"].to_numpy()
y_prob = pred["_y_prob"].to_numpy()
y_pred = pred["_y_pred"].to_numpy()

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

point_metrics = {
    "True negatives (TN)": float(tn),
    "False positives (FP)": float(fp),
    "False negatives (FN)": float(fn),
    "True positives (TP)": float(tp),
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred, zero_division=0),
    "Sensitivity (recall)": recall_score(y_true, y_pred, zero_division=0),
    "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
    "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
    "F1-score": f1_score(y_true, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_true, y_prob),
    "Average precision": average_precision_score(y_true, y_prob),
    "Brier score": brier_score_loss(y_true, y_prob),
}

display(pd.DataFrame(
    point_metrics.items(), columns=["Metric", "Value"]
))


In [ ]:

# ------------------------------------------------------------------
# 11) Subject-level bootstrap 95% CI
# ------------------------------------------------------------------
rng = np.random.default_rng(CONFIG["bootstrap_seed"])
B = CONFIG["bootstrap_iterations"]

bootstrap_rows = []

for b in range(B):
    idx = rng.integers(0, len(y_true), size=len(y_true))
    yt = y_true[idx]
    yp = y_prob[idx]
    yh = (yp >= CONFIG["operating_threshold"]).astype(int)

    if np.unique(yt).size < 2:
        continue

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(
        yt, yh, labels=[0, 1]
    ).ravel()

    bootstrap_rows.append({
        "Accuracy": accuracy_score(yt, yh),
        "Precision": precision_score(yt, yh, zero_division=0),
        "Sensitivity (recall)": recall_score(yt, yh, zero_division=0),
        "Specificity": (
            tn_b / (tn_b + fp_b)
            if (tn_b + fp_b) else np.nan
        ),
        "Balanced accuracy": balanced_accuracy_score(yt, yh),
        "F1-score": f1_score(yt, yh, zero_division=0),
        "ROC-AUC": roc_auc_score(yt, yp),
        "Average precision": average_precision_score(yt, yp),
        "Brier score": brier_score_loss(yt, yp),
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)

ci_map = {}
for metric in bootstrap_df.columns:
    ci_map[metric] = percentile_ci(bootstrap_df[metric])

table3_rows = []

for metric, value in point_metrics.items():
    if metric in {
        "True negatives (TN)", "False positives (FP)",
        "False negatives (FN)", "True positives (TP)"
    }:
        value_display = f"{int(value)}"
        ci_display = "—"
    else:
        value_display = f"{value:.3f}"
        low, high = ci_map.get(metric, (np.nan, np.nan))
        ci_display = (
            f"{low:.3f}–{high:.3f}"
            if np.isfinite(low) and np.isfinite(high)
            else "—"
        )

    table3_rows.append([metric, value_display, ci_display])

table3 = pd.DataFrame(
    table3_rows,
    columns=["Metric", "Value", "Bootstrap 95% CI"],
)

display(table3)


In [ ]:

# ------------------------------------------------------------------
# 12) Statistical analysis manifest
# ------------------------------------------------------------------
manifest = {
    "prediction_file": str(PREDICTION_FILE),
    "clinical_file": str(CLINICAL_FILE),
    "output_directory": str(OUTPUT_DIR),
    "prediction_columns": {
        "subject_id": pred_id_col,
        "ground_truth": true_col,
        "probability": prob_col,
        "prediction": pred_label_col,
        "slice_count": slice_count_col,
    },
    "clinical_columns": clinical_cols,
    "cohort": {
        "labelled_subjects": int(len(pred)),
        "CN": int((pred["_group"] == "CN").sum()),
        "AD": int((pred["_group"] == "AD").sum()),
        "matched_clinical_records": int(
            (merged["_merge"] == "both").sum()
        ),
        "unmatched_subjects": int(
            (merged["_merge"] != "both").sum()
        ),
    },
    "operating_threshold": CONFIG["operating_threshold"],
    "bootstrap_iterations": CONFIG["bootstrap_iterations"],
    "bootstrap_seed": CONFIG["bootstrap_seed"],
    "environment": environment,
    "model_configuration": CONFIG,
    "point_metrics": {
        key: float(value)
        for key, value in point_metrics.items()
    },
}

print(json.dumps(manifest, ensure_ascii=False, indent=2)[:5000])


In [ ]:

# ------------------------------------------------------------------
# 13) Export CSV, Excel, and JSON
# ------------------------------------------------------------------
table1.to_csv(
    OUTPUT_DIR / "Table1_Demographic_Clinical_Characteristics.csv",
    index=False,
    encoding="utf-8-sig",
)
table2.to_csv(
    OUTPUT_DIR / "Table2_Model_Configuration.csv",
    index=False,
    encoding="utf-8-sig",
)
table3.to_csv(
    OUTPUT_DIR / "Table3_Performance_with_Bootstrap_CI.csv",
    index=False,
    encoding="utf-8-sig",
)
merge_audit.to_csv(
    OUTPUT_DIR / "Cohort_Merge_Audit.csv",
    index=False,
    encoding="utf-8-sig",
)
unmatched.to_csv(
    OUTPUT_DIR / "Unmatched_Subjects.csv",
    index=False,
    encoding="utf-8-sig",
)
missing_report.to_csv(
    OUTPUT_DIR / "Missing_Data_Report.csv",
    index=False,
    encoding="utf-8-sig",
)
bootstrap_df.to_csv(
    OUTPUT_DIR / "Bootstrap_Distributions.csv",
    index=False,
)

excel_path = OUTPUT_DIR / "BrainFMOps_Publication_Tables.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    table1.to_excel(writer, sheet_name="Table 1", index=False)
    table2.to_excel(writer, sheet_name="Table 2", index=False)
    table3.to_excel(writer, sheet_name="Table 3", index=False)
    merge_audit.to_excel(writer, sheet_name="Merge Audit", index=False)
    unmatched.to_excel(writer, sheet_name="Unmatched", index=False)
    missing_report.to_excel(writer, sheet_name="Missing Data", index=False)
    bootstrap_df.to_excel(
        writer, sheet_name="Bootstrap", index=False
    )

with open(
    OUTPUT_DIR / "Statistical_Analysis_Manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("=" * 78)
print("PUBLICATION TABLE PACKAGE CREATED")
print("OUTPUT:", OUTPUT_DIR)
print("EXCEL :", excel_path)
print("=" * 78)


In [ ]:

# ------------------------------------------------------------------
# 14) Final validation — stop if critical values are inconsistent
# ------------------------------------------------------------------
checks = {
    "Expected labelled subjects = 212": len(pred) == 212,
    "Expected CN = 124": int((pred["_group"] == "CN").sum()) == 124,
    "Expected AD = 88": int((pred["_group"] == "AD").sum()) == 88,
    "Expected TN = 36": int(tn) == 36,
    "Expected FP = 88": int(fp) == 88,
    "Expected FN = 15": int(fn) == 15,
    "Expected TP = 73": int(tp) == 73,
}

validation = pd.DataFrame(
    [{"Check": k, "Pass": bool(v)} for k, v in checks.items()]
)

display(validation)

if not validation["Pass"].all():
    print(
        "\nWARNING: ผลบางค่าต่างจากต้นฉบับ "
        "กรุณาหยุดและตรวจไฟล์ input ก่อนนำตารางไปใช้"
    )
else:
    print(
        "\nPASS: cohort และ confusion matrix "
        "ตรงกับผลที่ใช้ในต้นฉบับ"
    )



## ไฟล์ผลลัพธ์

```text
34_Table_Generator_Output
├── BrainFMOps_Publication_Tables.xlsx
├── Table1_Demographic_Clinical_Characteristics.csv
├── Table2_Model_Configuration.csv
├── Table3_Performance_with_Bootstrap_CI.csv
├── Cohort_Merge_Audit.csv
├── Unmatched_Subjects.csv
├── Missing_Data_Report.csv
├── Bootstrap_Distributions.csv
└── Statistical_Analysis_Manifest.json
```

## ก่อนนำ Table 1 ไปใช้

เกณฑ์ที่ดีที่สุดคือ:

```text
Matched clinical records = 212
Unmatched subjects = 0
Duplicate clinical IDs = 0
```

หาก merge ไม่ครบ 212 ราย ค่า demographic จะไม่ใช่ cohort เดียวกับ confusion matrix

## ก่อนนำ Table 2 ไปใช้

ช่องที่ขึ้นต้นด้วย `UNKNOWN` ต้องเติมจาก inference/training code จริง ได้แก่:

- model architecture
- pretraining corpus
- checkpoint
- classification head
- input resolution/channels
- normalization
- slice-selection rule

ห้ามเดาข้อมูลเหล่านี้ เพราะเป็นแกนหลักของ reproducibility claim
